# BANK MARKETING — Construire un pipeline complet de Machine Learning

### De la question métier à un modèle évalué sans fuite de données

**Formation Machine Learning · Notebook de référence et support pratique**

---

## Fil conducteur

Une banque souhaite mieux cibler ses campagnes de marketing direct. À partir d'informations disponibles sur les clients et sur les campagnes précédentes, peut-on estimer si un client souscrira à un dépôt à terme ?

Ce notebook construit une démarche de Machine Learning reproductible :

1. cadrage du problème et formulation de l'hypothèse ;
2. acquisition et audit des données ;
3. analyse exploratoire (EDA) ;
4. définition du scénario de prédiction et prévention des fuites ;
5. séparation entraînement / validation / test ;
6. prétraitement numérique et catégoriel ;
7. baseline et comparaison de modèles ;
8. validation croisée et recherche d'hyperparamètres ;
9. évaluation finale et analyse des erreurs ;
10. interprétation, limites et mise en situation métier.

**Principe directeur :** une bonne démarche ne consiste pas à obtenir le score le plus élevé possible à tout prix. Elle consiste à estimer honnêtement la performance attendue dans un scénario de déploiement défini.

> **Attention à la portée des données.** Le dataset UCI décrit des campagnes téléphoniques d'une banque portugaise entre mai 2008 et novembre 2010. Il est réel et documenté, mais ancien : ce n'est pas une représentation garantie des comportements bancaires actuels. Source : UCI Machine Learning Repository, Bank Marketing.

## 0. Objectifs pédagogiques et prérequis

À la fin du notebook, vous devriez pouvoir :

- distinguer une cible, des variables explicatives et des informations indisponibles au moment de la décision ;
- conduire une EDA utile à la modélisation, plutôt qu'une simple collection de graphiques ;
- choisir une séparation adaptée à la structure temporelle des données ;
- construire un pipeline avec `ColumnTransformer` et `Pipeline` ;
- comparer un modèle linéaire à des modèles à base d'arbres ;
- expliquer pourquoi l'accuracy seule est insuffisante en présence d'une classe minoritaire ;
- séparer la sélection du modèle de son évaluation finale.

### Environnement

Bibliothèques utilisées : `pandas`, `numpy`, `matplotlib`, `seaborn` et `scikit-learn`.

Si une bibliothèque manque dans votre environnement, installez-la avant la séance, par exemple avec `pip install pandas numpy matplotlib seaborn scikit-learn`.

In [ ]:
import sys
import warnings
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay, precision_recall_curve
)

warnings.filterwarnings("default")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)

## 1. Cadrage métier : quelle décision veut-on améliorer ?

### 1.1 Problème métier

La cible `y` indique si le client a souscrit (`yes`) ou non (`no`) à un dépôt à terme à l'issue de la campagne. Le problème est donc une **classification binaire supervisée**.

La prédiction peut aider à prioriser des contacts, mais elle ne doit pas être interprétée automatiquement comme une estimation de l'effet causal d'un appel. Un client susceptible de souscrire n'est pas nécessairement un client dont la souscription est provoquée par la campagne.

### 1.2 Traduire la question en formulation ML

- **Unité observée :** une ligne du jeu de données, représentant un contact / dossier de campagne selon la version.
- **Cible :** `y`.
- **Classe positive :** `yes`.
- **Moment de prédiction envisagé :** avant l'appel de la campagne en cours.
- **Conséquence méthodologique :** une variable mesurée pendant ou après l'appel ne doit pas être utilisée pour prétendre prédire avant l'appel.

### 1.3 Une question à garder en tête

> Si un modèle obtient 95 % d'accuracy mais ne retrouve presque aucun client qui souscrit, est-il utile ?

Nous reviendrons sur cette question lors de l'évaluation.

## 2. Acquisition des données et traçabilité

Nous téléchargeons l'archive officielle UCI. Le notebook recherche ensuite le fichier `bank-additional-full.csv` (version enrichie de 41 188 observations) dans l'archive.

Source officielle : https://archive.ics.uci.edu/dataset/222/bank+marketing

**Reproductibilité :** la source, le fichier retenu et les dimensions sont affichés explicitement. Si le téléchargement échoue dans votre environnement, téléchargez l'archive depuis le site UCI et adaptez `LOCAL_CSV_PATH`.

In [ ]:
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
LOCAL_CSV_PATH = None  # Exemple : "data/bank-additional-full.csv"

def load_bank_marketing(local_csv_path=None):
    if local_csv_path is not None and Path(local_csv_path).exists():
        return pd.read_csv(local_csv_path, sep=";"), f"fichier local : {local_csv_path}"

    try:
        with urlopen(UCI_ZIP_URL, timeout=30) as response:
            archive_bytes = response.read()
        with ZipFile(BytesIO(archive_bytes)) as zf:
            candidates = [
                name for name in zf.namelist()
                if name.lower().endswith("bank-additional-full.csv")
            ]
            if not candidates:
                candidates = [
                    name for name in zf.namelist()
                    if name.lower().endswith("bank-full.csv")
                ]
            if not candidates:
                raise FileNotFoundError("Aucun CSV Bank Marketing reconnu dans l'archive.")
            selected = candidates[0]
            with zf.open(selected) as file:
                frame = pd.read_csv(file, sep=";")
        return frame, f"archive UCI : {selected}"
    except Exception as exc:
        raise RuntimeError(
            "Téléchargement impossible. Téléchargez l'archive depuis "
            "https://archive.ics.uci.edu/dataset/222/bank+marketing puis "
            "renseignez LOCAL_CSV_PATH."
        ) from exc

df_raw, data_source = load_bank_marketing(LOCAL_CSV_PATH)
print("Source :", data_source)
print("Dimensions :", df_raw.shape)
display(df_raw.head())

### 2.1 Premier contrôle du schéma

Avant de modéliser, il faut vérifier que les données chargées correspondent bien à ce que l'on pense avoir chargé.

In [ ]:
print("Nombre de lignes :", len(df_raw))
print("Nombre de colonnes :", df_raw.shape[1])
print("\nNoms des colonnes :")
print(df_raw.columns.tolist())
print("\nTypes :")
display(df_raw.dtypes.rename("dtype").to_frame())

## 3. Audit de qualité des données

L'audit initial doit répondre à quatre questions :

1. Les types de données sont-ils cohérents ?
2. Existe-t-il des valeurs manquantes explicites ou encodées sous forme de texte (`unknown`) ?
3. Y a-t-il des doublons exacts ?
4. Les valeurs et modalités sont-elles plausibles ?

**Important :** dans ce dataset, `unknown` est souvent une modalité explicite et non un NaN. Ce n'est pas automatiquement une erreur. Il faut décider si cette information est informative, si elle représente une absence de renseignement, ou si son sens dépend du processus métier.

In [ ]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_missing": df_raw.isna().sum(),
    "missing_rate_%": (100 * df_raw.isna().mean()).round(2),
    "n_unique": df_raw.nunique(dropna=False)
}).sort_values(["missing_rate_%", "n_unique"], ascending=[False, True])

display(audit)
print("Doublons exacts :", int(df_raw.duplicated().sum()))

unknown_summary = []
for col in df_raw.select_dtypes(include="object").columns:
    count = df_raw[col].astype(str).str.strip().str.lower().eq("unknown").sum()
    if count:
        unknown_summary.append({"variable": col, "unknown_count": int(count),
                                "unknown_rate_%": round(100 * count / len(df_raw), 2)})
display(pd.DataFrame(unknown_summary).sort_values("unknown_rate_%", ascending=False))

### Questions d'analyse

- Un taux de `unknown` élevé signifie-t-il nécessairement qu'il faut supprimer la variable ?
- Pourquoi ne faut-il pas remplacer automatiquement toutes les modalités `unknown` par la modalité majoritaire ?
- Deux lignes identiques sont-elles toujours des erreurs ? Dans un historique de contacts, des événements répétés peuvent avoir une signification.

## 4. Comprendre les variables et le mécanisme de collecte

La version `bank-additional-full.csv` contient des informations de plusieurs natures :

| Famille | Exemples | Interprétation |
|---|---|---|
| Profil client | `age`, `job`, `marital`, `education` | Informations générales |
| Situation financière | `default`, `housing`, `loan` | Indicateurs de crédit / prêts |
| Contact courant | `contact`, `month`, `day_of_week`, `duration` | Informations sur l'appel |
| Historique de campagne | `campaign`, `pdays`, `previous`, `poutcome` | Contacts précédents |
| Contexte macroéconomique | `emp.var.rate`, `cons.price.idx`, `euribor3m`, etc. | Indicateurs économiques |
| Cible | `y` | Souscription au dépôt à terme |

Les variables de contact courant et les variables historiques exigent une attention particulière : leur validité dépend du moment exact où la prédiction est produite.

In [ ]:
variable_dictionary = pd.DataFrame([
    ("age", "Profil client", "Âge du client"),
    ("job", "Profil client", "Catégorie professionnelle"),
    ("marital", "Profil client", "Situation matrimoniale"),
    ("education", "Profil client", "Niveau d'études"),
    ("default", "Situation financière", "Crédit en défaut selon la variable"),
    ("housing", "Situation financière", "Prêt immobilier"),
    ("loan", "Situation financière", "Prêt personnel"),
    ("contact", "Contact courant", "Canal de contact"),
    ("month", "Contact courant", "Mois du dernier contact"),
    ("day_of_week", "Contact courant", "Jour de la semaine du dernier contact"),
    ("duration", "Contact courant", "Durée de l'appel, connue après l'appel"),
    ("campaign", "Campagne courante", "Nombre de contacts pendant la campagne"),
    ("pdays", "Historique", "Jours depuis un contact précédent ; 999 signifie souvent aucun contact antérieur dans cette version"),
    ("previous", "Historique", "Nombre de contacts antérieurs"),
    ("poutcome", "Historique", "Résultat de la campagne précédente"),
    ("emp.var.rate", "Macroéconomie", "Variation du taux d'emploi"),
    ("cons.price.idx", "Macroéconomie", "Indice des prix à la consommation"),
    ("cons.conf.idx", "Macroéconomie", "Indice de confiance des consommateurs"),
    ("euribor3m", "Macroéconomie", "Taux Euribor à trois mois"),
    ("nr.employed", "Macroéconomie", "Indicateur du nombre de personnes employées"),
    ("y", "Cible", "Souscription au dépôt à terme")
], columns=["variable", "famille", "description"])

display(variable_dictionary)

## 5. Définir un scénario de prédiction réaliste — prévenir la fuite de données

La fuite de données (*data leakage*) survient lorsqu'un modèle utilise une information qui ne serait pas disponible au moment réel de la prédiction, ou qui incorpore indirectement la réponse.

### 5.1 Le cas critique : `duration`

La documentation UCI précise que `duration` correspond à la durée du dernier appel. Cette durée n'est pas connue avant l'appel et elle est fortement liée à la cible. Si l'objectif est de décider **avant d'appeler** qui contacter, l'inclure rend le problème irréaliste.

Nous allons donc :
- exclure `duration` du scénario principal ;
- documenter cette exclusion ;
- ne jamais interpréter une performance obtenue avec `duration` comme une performance pré-appel.

### 5.2 Autres risques

- `poutcome` et les variables historiques ne sont utilisables que si elles étaient disponibles au moment de la décision.
- Les variables macroéconomiques peuvent être redondantes et fortement corrélées.
- Les données sont ordonnées par date dans la version complète, mais ne fournissent pas nécessairement une date de contact exacte utilisable pour une validation temporelle parfaite.

**Choix principal du notebook :** nous excluons `duration` et nous conservons les variables restantes pour un pipeline pédagogique. Nous comparerons une séparation aléatoire stratifiée avec une réflexion sur les limites temporelles. Pour une estimation stricte de déploiement futur, une validation temporelle documentée serait préférable.

In [ ]:
TARGET = "y"
LEAKAGE_FEATURES = ["duration"]

# Cible explicite : yes -> 1, no -> 0
y = df_raw[TARGET].map({"yes": 1, "no": 0})
if y.isna().any():
    raise ValueError("La cible contient des modalités non prévues ; vérifier le schéma.")

X = df_raw.drop(columns=[TARGET] + [c for c in LEAKAGE_FEATURES if c in df_raw.columns]).copy()
y = y.astype("int8")

print("X :", X.shape)
print("y :", y.shape)
print("Variables exclues pour le scénario pré-appel :", LEAKAGE_FEATURES)
print("Taux de souscription :", f"{y.mean():.2%}")
display(y.value_counts().rename(index={0: "no", 1: "yes"}).to_frame("effectif"))

## 6. Analyse exploratoire des données (EDA)

L'EDA n'est pas une étape décorative. Elle sert à formuler des hypothèses, détecter des problèmes de qualité et choisir des traitements. Les graphiques doivent être interprétés avec prudence : une association observée ne prouve pas un effet causal.

Nous explorons :
1. la variable cible ;
2. les distributions numériques ;
3. les variables catégorielles ;
4. les taux de souscription par groupe ;
5. la structure des valeurs inconnues et des variables de campagne.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(x=y.map({0: "No", 1: "Yes"}), ax=axes[0])
axes[0].set_title("Répartition de la cible")
axes[0].set_xlabel("Souscription")
axes[0].set_ylabel("Nombre d'observations")

class_share = y.value_counts(normalize=True).sort_index()
axes[1].bar(["No", "Yes"], [class_share.get(0, 0), class_share.get(1, 0)])
axes[1].set_title("Proportion de chaque classe")
axes[1].set_ylabel("Proportion")
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

print(f"Classe majoritaire : {max(y.mean(), 1-y.mean()):.1%}")
print(f"Baseline naïve d'accuracy si l'on prédit toujours la classe majoritaire : {max(y.mean(), 1-y.mean()):.1%}")

### 6.1 Distributions des variables numériques

Les valeurs extrêmes ne doivent pas être supprimées mécaniquement. Il faut comprendre si elles sont impossibles, rares mais plausibles, ou liées à la collecte.

In [ ]:
numeric_all = X.select_dtypes(include=np.number).columns.tolist()
print("Variables numériques :", numeric_all)

X[numeric_all].hist(figsize=(16, 12), bins=30, edgecolor="white")
plt.suptitle("Distributions des variables numériques", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

### 6.2 Catégories et modalités rares

Les variables catégorielles peuvent comporter de nombreuses modalités. `unknown` est conservé comme modalité distincte dans le pipeline initial.

In [ ]:
categorical_all = X.select_dtypes(exclude=np.number).columns.tolist()
print("Variables catégorielles :", categorical_all)

for col in categorical_all:
    display(
        X[col].value_counts(dropna=False)
        .head(15)
        .rename_axis(col)
        .to_frame("count")
    )

### 6.3 Taux de souscription par catégorie

Ces taux sont descriptifs. Ils peuvent varier avec l'âge, la saison, la stratégie de campagne et d'autres variables confondantes.

In [ ]:
for col in ["job", "education", "marital", "contact", "month", "poutcome"]:
    if col in df_raw.columns:
        rates = (
            df_raw.assign(_target=y)
            .groupby(col, dropna=False)["_target"]
            .agg(["mean", "count"])
            .sort_values("mean", ascending=False)
        )
        rates["mean_pct"] = 100 * rates["mean"]
        display(rates[["mean_pct", "count"]].round(2).style.set_caption(f"Taux observé par {col}"))

### 6.4 Corrélations entre variables numériques

La corrélation mesure une association linéaire, pas une causalité. Pour les indicateurs macroéconomiques, une forte corrélation peut rendre les coefficients d'un modèle linéaire moins stables sans empêcher nécessairement une bonne prédiction.

In [ ]:
if len(numeric_all) >= 2:
    corr = X[numeric_all].corr(numeric_only=True)
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr, cmap="vlag", center=0, annot=False)
    plt.title("Corrélations entre variables numériques")
    plt.tight_layout()
    plt.show()

### 6.5 Synthèse de l'EDA

Avant de passer au modèle, rédigez trois observations factuelles et deux décisions de préparation des données. Par exemple :
- la classe positive est-elle minoritaire ?
- quelles variables ont des distributions très asymétriques ?
- quelles variables catégorielles ont beaucoup de modalités ?
- quelles variables sont susceptibles d'être disponibles uniquement après la décision ?

## 7. Définir les ensembles d'entraînement et de test

Le test doit représenter des observations qui n'ont pas servi à choisir les transformations, les modèles ou les hyperparamètres.

Pour l'exercice, nous utilisons une séparation stratifiée aléatoire. Elle préserve approximativement la proportion de la classe positive. **Elle ne simule pas parfaitement un déploiement dans le futur**, car la campagne d'origine possède une structure temporelle. En production, une séparation chronologique serait à privilégier si l'objectif est de généraliser à une campagne future.

Nous réservons 20 % des observations au test final. Nous n'utilisons pas ce test pendant la sélection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

split_summary = pd.DataFrame({
    "ensemble": ["Train", "Test"],
    "n_observations": [len(y_train), len(y_test)],
    "taux_souscription": [y_train.mean(), y_test.mean()]
})
display(split_summary.style.format({"taux_souscription": "{:.2%}"}))

## 8. Construire un prétraitement robuste

### Pourquoi utiliser `Pipeline` et `ColumnTransformer` ?

Les étapes de prétraitement peuvent apprendre des paramètres : médiane d'imputation, moyenne et écart-type, catégories rencontrées lors de l'encodage. Si ces paramètres sont calculés sur toutes les données avant la validation, des informations de validation ou de test contaminent l'entraînement.

Le pipeline garantit que chaque pli de validation ajuste ses transformations uniquement sur sa partie entraînement.

### Choix de traitement

- Numériques : imputation par médiane.
- Catégorielles : imputation par modalité la plus fréquente.
- Catégorielles : encodage one-hot avec tolérance aux catégories inconnues.
- Standardisation : appliquée aux variables numériques pour la régression logistique.
- Arbres : la standardisation n'est généralement pas nécessaire ; pour garder la comparaison lisible, nous utiliserons d'abord un prétraitement commun. Une amélioration possible consiste à créer un prétraitement spécifique aux modèles à base d'arbres.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_features),
        ("cat", categorical_preprocessor, categorical_features)
    ],
    remainder="drop"
)

print("Nombre de variables numériques :", len(numeric_features))
print("Nombre de variables catégorielles :", len(categorical_features))
print("Prétraitement défini ; aucune transformation n'a encore été ajustée.")

## 9. Définir une baseline et des modèles candidats

Une baseline n'est pas facultative : elle fournit un point de comparaison minimal. Si un modèle complexe ne fait pas mieux qu'une stratégie simple, il n'y a pas de raison de le préférer.

Nous comparons :
- `DummyClassifier` : prédit la classe majoritaire ;
- régression logistique : modèle linéaire, interprétable et souvent solide comme référence ;
- arbre de décision : capture des interactions et seuils non linéaires ;
- forêt aléatoire : agrégation d'arbres, souvent plus stable qu'un arbre isolé.

Les paramètres sont volontairement modérés pour tenir dans une séance de formation.

In [ ]:
models = {
    "Baseline majoritaire": DummyClassifier(strategy="most_frequent"),
    "Régression logistique": LogisticRegression(
        max_iter=2000, class_weight=None, random_state=RANDOM_STATE
    ),
    "Arbre de décision": DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=30, random_state=RANDOM_STATE
    ),
    "Forêt aléatoire": RandomForestClassifier(
        n_estimators=250, min_samples_leaf=5,
        n_jobs=-1, random_state=RANDOM_STATE
    )
}

pipelines = {
    name: Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])
    for name, estimator in models.items()
}

## 10. Choisir les métriques avant d'observer les résultats

Dans ce problème, la souscription est minoritaire. L'accuracy peut être trompeuse.

- **Precision** : parmi les clients prédits positifs, quelle proportion souscrit réellement ?
- **Recall** : parmi les clients qui souscrivent, quelle proportion le modèle retrouve-t-il ?
- **F1** : moyenne harmonique de precision et recall.
- **ROC-AUC** : capacité de classement sur différents seuils, sans fixer un seuil unique.
- **Average Precision** : résumé de la courbe précision-rappel, souvent informatif lorsque la classe positive est rare.
- **Balanced accuracy** : moyenne du recall de chaque classe.

Le meilleur choix dépend du coût métier des faux positifs et faux négatifs. Pour cette démonstration, nous suivons F1 et Average Precision en complément de l'accuracy.

In [ ]:
SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Validation croisée :", cv)
print("Métrique principale de sélection dans cette démonstration : F1")

## 11. Comparaison initiale par validation croisée

Nous calculons les métriques sur les plis de validation du jeu d'entraînement. Le jeu de test reste intact.

In [ ]:
cv_rows = []
cv_raw_results = {}

for name, pipe in pipelines.items():
    result = cross_validate(
        pipe,
        X_train, y_train,
        cv=cv,
        scoring=SCORING,
        return_train_score=False,
        n_jobs=-1,
        error_score="raise"
    )
    cv_raw_results[name] = result
    row = {"modèle": name}
    for metric in SCORING:
        values = result[f"test_{metric}"]
        row[f"{metric}_moyenne"] = values.mean()
        row[f"{metric}_écart_type"] = values.std(ddof=1)
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("f1_moyenne", ascending=False)
display(cv_results.round(3))

In [ ]:
plot_data = cv_results.melt(
    id_vars="modèle",
    value_vars=["accuracy_moyenne", "precision_moyenne", "recall_moyenne", "f1_moyenne"],
    var_name="métrique", value_name="score"
)
plt.figure(figsize=(13, 5))
sns.barplot(data=plot_data, x="modèle", y="score", hue="métrique")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.title("Comparaison des modèles — moyenne de validation croisée")
plt.tight_layout()
plt.show()

### Discussion : quel modèle choisir ?

Ne retenez pas uniquement le maximum numérique. Discutez :
- du compromis precision / recall ;
- de la variabilité entre plis ;
- de la complexité et de l'explicabilité ;
- de l'écart avec la baseline ;
- de la métrique réellement alignée sur l'objectif métier.

Une performance moyenne proche peut être préférable si le modèle est plus simple, plus stable ou plus facile à expliquer.

## 12. Sélection et réglage des hyperparamètres

Nous illustrons `GridSearchCV` avec la forêt aléatoire. Les paramètres sont réglés exclusivement sur `X_train`. Le pipeline complet est évalué dans chaque pli : le prétraitement est donc ajusté à l'intérieur du pli.

Pour limiter le temps de calcul, la grille est volontairement compacte. Dans un vrai projet, le choix des paramètres, la stratégie de validation et le budget de calcul doivent être justifiés.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [150, 300],
    "model__max_depth": [None, 10, 18],
    "model__min_samples_leaf": [2, 8],
    "model__max_features": ["sqrt", 0.7]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Meilleurs paramètres :")
print(grid_search.best_params_)
print("Meilleur F1 moyen en CV :", round(grid_search.best_score_, 4))

best_model = grid_search.best_estimator_

In [ ]:
grid_results = pd.DataFrame(grid_search.cv_results_)
display(
    grid_results[
        ["rank_test_score", "mean_test_score", "std_test_score",
         "mean_train_score", "param_model__n_estimators",
         "param_model__max_depth", "param_model__min_samples_leaf",
         "param_model__max_features"]
    ].sort_values("rank_test_score").head(10).round(4)
)

### Lire les résultats de recherche

- Un score d'entraînement bien supérieur au score de validation peut signaler du surapprentissage.
- Un écart-type élevé indique que les résultats dépendent davantage du pli utilisé.
- Une grille plus large augmente le coût de calcul et peut augmenter le risque de suradapter les choix au protocole de validation.
- La validation croisée ne remplace pas le test final.

## 13. Évaluation finale — une fois les choix terminés

Le meilleur pipeline de la recherche a été réajusté sur tout `X_train` par `GridSearchCV(refit=True)`. Nous l'évaluons maintenant sur `X_test`, jamais utilisé pour choisir les hyperparamètres.

Les résultats ci-dessous sont une estimation sur ce jeu de test particulier. Ils ne constituent pas une garantie de performance future.

In [ ]:
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

final_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred, zero_division=0),
    "F1": f1_score(y_test, y_test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_test_proba),
    "Average precision": average_precision_score(y_test, y_test_proba)
}

display(pd.Series(final_metrics, name="score").to_frame().round(4))
print("\nRapport de classification :")
print(classification_report(
    y_test, y_test_pred,
    target_names=["No subscription", "Subscription"],
    zero_division=0
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred,
    display_labels=["No", "Yes"],
    cmap="Blues",
    ax=axes[0],
    colorbar=False
)
axes[0].set_title("Matrice de confusion — test")

RocCurveDisplay.from_predictions(y_test, y_test_proba, ax=axes[1])
axes[1].set_title("Courbe ROC — test")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
PrecisionRecallDisplay.from_predictions(y_test, y_test_proba)
plt.title("Courbe précision-rappel — test")
plt.tight_layout()
plt.show()

## 14. Le seuil de décision : 0,5 n'est pas une loi

Par défaut, `predict()` utilise généralement un seuil de probabilité de 0,5 pour la classe positive. Ce seuil n'est pas forcément adapté au contexte métier.

- Un seuil plus bas tend à augmenter le recall, mais peut augmenter les faux positifs.
- Un seuil plus élevé tend à augmenter la precision, mais peut manquer davantage de souscriptions.

Le seuil doit être choisi sur les données de validation en fonction d'un objectif défini. **Il ne faut pas le régler sur le test puis présenter les résultats sur ce même test comme une évaluation indépendante.**

L'exemple suivant visualise les compromis de seuil sur les probabilités du test uniquement à titre exploratoire ; ces valeurs ne doivent pas être utilisées pour sélectionner le seuil final de production.

In [ ]:
threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.05):
    pred_at_threshold = (y_test_proba >= threshold).astype(int)
    threshold_rows.append({
        "seuil": threshold,
        "precision": precision_score(y_test, pred_at_threshold, zero_division=0),
        "recall": recall_score(y_test, pred_at_threshold, zero_division=0),
        "f1": f1_score(y_test, pred_at_threshold, zero_division=0),
        "predits_positifs": int(pred_at_threshold.sum())
    })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results.round(3))

In [ ]:
plt.figure(figsize=(9, 5))
for metric in ["precision", "recall", "f1"]:
    plt.plot(threshold_results["seuil"], threshold_results[metric], marker="o", label=metric)
plt.xlabel("Seuil de probabilité")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Compromis precision / recall / F1 selon le seuil — analyse illustrative")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Comprendre les erreurs du modèle

La matrice de confusion distingue :
- vrais négatifs (TN) ;
- faux positifs (FP) ;
- faux négatifs (FN) ;
- vrais positifs (TP).

Dans ce cas métier, un faux positif peut conduire à contacter un client qui ne souscrira pas ; un faux négatif peut correspondre à un client potentiel qui n'est pas priorisé. Le coût réel dépend de la stratégie de campagne et du coût d'un contact.

Il ne suffit donc pas de maximiser F1 sans discuter des coûts et de la capacité de contact.

In [ ]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred, labels=[0, 1]).ravel()
error_summary = pd.DataFrame({
    "type": ["Vrais négatifs (TN)", "Faux positifs (FP)", "Faux négatifs (FN)", "Vrais positifs (TP)"],
    "effectif": [tn, fp, fn, tp],
    "interprétation": [
        "Non-souscriptions correctement identifiées",
        "Clients ciblés par le modèle qui ne souscrivent pas",
        "Souscriptions manquées par le modèle",
        "Souscriptions correctement identifiées"
    ]
})
display(error_summary)

## 16. Interprétation des variables : importance de la forêt aléatoire

L'importance par permutation mesure la baisse de performance lorsqu'on mélange une variable. Elle est calculée ici sur le jeu de test, donc elle est **diagnostique et postérieure à l'évaluation** : elle ne doit pas être utilisée pour recommencer la sélection du modèle tout en prétendant garder le test intact.

Pour une analyse plus rigoureuse, calculez les importances sur un jeu de validation distinct ou utilisez une procédure de validation imbriquée. Une importance élevée ne signifie pas causalité.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
perm_importance = pd.DataFrame({
    "variable": X_test.columns,
    "importance_moyenne": perm.importances_mean,
    "importance_ecart_type": perm.importances_std
}).sort_values("importance_moyenne", ascending=False)

display(perm_importance.head(15).round(4))

plt.figure(figsize=(9, 6))
top = perm_importance.head(15).sort_values("importance_moyenne")
plt.barh(top["variable"], top["importance_moyenne"], xerr=top["importance_ecart_type"])
plt.xlabel("Baisse d'Average Precision après permutation")
plt.title("Importance par permutation — diagnostic sur test")
plt.tight_layout()
plt.show()

## 17. Retour critique : que signifie réellement le résultat ?

Avant de conclure, vérifiez ces limites :

1. **Données anciennes :** campagne portugaise de 2008–2010 ; les habitudes, canaux et conditions économiques ont changé.
2. **Échantillonnage et temporalité :** une séparation aléatoire n'est pas identique à une prédiction sur une campagne future.
3. **Variables disponibles :** les caractéristiques doivent être disponibles au moment réel de la décision.
4. **Déséquilibre de classe :** l'accuracy peut cacher un recall faible pour la classe positive.
5. **Dérive des données :** une mise en production nécessite de surveiller les distributions et les performances dans le temps.
6. **Éthique et réglementation :** une utilisation bancaire réelle exige une analyse de conformité, de confidentialité, d'équité et de justification des décisions.
7. **Prédiction vs causalité :** le modèle estime une association prédictive ; il n'identifie pas nécessairement les clients dont la décision changera grâce à un appel.

Un pipeline propre est nécessaire, mais ne suffit pas à rendre le modèle opérationnel.

## 18. Exercices d'approfondissement

### Exercice A — Changer la métrique
Répétez la recherche d'hyperparamètres avec `average_precision` comme métrique de sélection. Comparez le modèle retenu et expliquez pourquoi le classement peut changer.

### Exercice B — Modèles avec et sans variables macroéconomiques
Retirez `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m` et `nr.employed`. Comparez les résultats. Vérifiez si le gain du modèle initial est robuste et discutez de la redondance des variables.

### Exercice C — Traitement des valeurs `unknown`
Comparez deux stratégies : conserver `unknown` comme modalité et remplacer `unknown` par une valeur manquante puis imputer. Gardez le reste du protocole identique.

### Exercice D — Seuil de décision
À partir de prédictions sur un jeu de validation distinct, choisissez un seuil pour atteindre un recall cible ou maximiser une fonction de coût métier. Évaluez ensuite une seule fois sur le test.

### Exercice E — Validation temporelle
La version complète est ordonnée par date de campagne. Proposez une séparation chronologique et expliquez pourquoi elle peut mieux représenter une mise en production future. Documentez précisément quelles lignes correspondent à chaque période avant d'appliquer cette stratégie.

### Exercice F — Coût de campagne
Supposez qu'un appel coûte \(c\), qu'une souscription rapporte une marge \(m\), et que le modèle ne peut sélectionner que \(k\) clients. Définissez une règle de sélection et une métrique économique. N'inventez pas les valeurs de \(c\) et \(m\) : traitez-les comme des hypothèses de scénario.

## 19. Synthèse finale

### Le pipeline que vous devez savoir reconstruire

1. **Définir** la question métier, la cible et le moment de prédiction.
2. **Collecter** les données et documenter leur provenance.
3. **Auditer** le schéma, les valeurs manquantes, les modalités et les doublons.
4. **Explorer** les distributions et les relations avec la cible.
5. **Prévenir les fuites** : supprimer les informations indisponibles au moment de la décision.
6. **Séparer** les données avant d'ajuster les transformations.
7. **Construire** le prétraitement dans un `Pipeline`.
8. **Établir une baseline** puis comparer plusieurs modèles avec une validation cohérente.
9. **Régler** les hyperparamètres uniquement sur les données d'entraînement.
10. **Évaluer** une fois sur le test final et interpréter les erreurs.
11. **Documenter les limites** avant toute conclusion ou mise en production.

### Référence du dataset

Moro, S., Rita, P., & Cortez, P. (2014). *A Data-Driven Approach to Predict the Success of Bank Telemarketing*. Decision Support Systems.

UCI Machine Learning Repository: [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), DOI: 10.24432/C5K306. Licence indiquée par UCI : CC BY 4.0.

---

**Message essentiel :** la rigueur du protocole de validation est aussi importante que le choix de l'algorithme.